# 03 — Counterfactual Optimization & Evaluation
### Compare candidate edits against a fixed baseline

**Purpose:** build the experiment loop that connects model analysis to safe optimization. This notebook demonstrates:
- a reproducible baseline
- gradient and neuron-level diagnostics
- candidate weight interventions
- evaluation on separate validation data
- simple acceptance gates
- experiment manifests and artifact hashes

This is a small synthetic MLP lab, not a claim that the same simple interventions directly transfer to large language models.

In [1]:
import copy
import json
import hashlib
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 23
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


## 1. Create a reproducible classification task

We use separate train, validation, and test sets. Candidate selection uses validation data; the test set is reserved for final reporting.

In [2]:
def make_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, 8, generator=g)
    score = 1.2*x[:, 0] - 0.8*x[:, 1] + 0.5*x[:, 2]*x[:, 3]
    y = torch.where(score > 0, 1, 0).long()
    return x, y

x_train, y_train = make_data(2500, 1)
x_val, y_val = make_data(700, 2)
x_test, y_test = make_data(1000, 3)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)

In [3]:
class Classifier(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(8, hidden)
        self.fc2 = nn.Linear(hidden, 2)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

def train_model(model, loader, epochs=12, lr=1e-2):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model

def evaluate(model, x, y):
    model.eval()
    x, y = x.to(DEVICE), y.to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        loss = F.cross_entropy(logits, y).item()
        accuracy = (logits.argmax(1) == y).float().mean().item()
    return {"loss": loss, "accuracy": accuracy}

baseline = train_model(Classifier().to(DEVICE), train_loader)
print("Train:", evaluate(baseline, x_train, y_train))
print("Validation:", evaluate(baseline, x_val, y_val))
print("Test:", evaluate(baseline, x_test, y_test))

Train: {'loss': 0.030535364523530006, 'accuracy': 0.9911999702453613}
Validation: {'loss': 0.07294617593288422, 'accuracy': 0.9785714149475098}
Test: {'loss': 0.08984179049730301, 'accuracy': 0.9750000238418579}


## 2. Rank hidden units by validation ablation impact

This is a group-level intervention: remove one hidden unit's outgoing contribution and measure the validation loss change. It is a screening method, not a proof of global redundancy.

In [4]:
@torch.no_grad()
def ablate_unit(model, unit_index):
    candidate = copy.deepcopy(model)
    candidate.fc2.weight[:, unit_index].zero_()
    return candidate

baseline_val = evaluate(baseline, x_val, y_val)
unit_effects = []
for unit in range(baseline.fc1.out_features):
    candidate = ablate_unit(baseline, unit)
    result = evaluate(candidate, x_val, y_val)
    unit_effects.append({
        "unit": unit,
        "delta_loss": result["loss"] - baseline_val["loss"],
        "delta_accuracy": result["accuracy"] - baseline_val["accuracy"],
    })

unit_effects_sorted = sorted(unit_effects, key=lambda row: row["delta_loss"])
unit_effects_sorted[:10]

[{'unit': 28,
  'delta_loss': -0.006384141743183136,
  'delta_accuracy': 0.0014286041259765625},
 {'unit': 24,
  'delta_loss': -0.006362617015838623,
  'delta_accuracy': 0.007142841815948486},
 {'unit': 5,
  'delta_loss': -0.003793083131313324,
  'delta_accuracy': -0.0014285445213317871},
 {'unit': 17,
  'delta_loss': -0.0028959959745407104,
  'delta_accuracy': -0.0028571486473083496},
 {'unit': 20,
  'delta_loss': -0.0024412870407104492,
  'delta_accuracy': -0.0014285445213317871},
 {'unit': 21,
  'delta_loss': -0.0021620988845825195,
  'delta_accuracy': -0.0028571486473083496},
 {'unit': 7,
  'delta_loss': -0.0020908862352371216,
  'delta_accuracy': 0.0014286041259765625},
 {'unit': 30,
  'delta_loss': -0.0013067424297332764,
  'delta_accuracy': -0.004285693168640137},
 {'unit': 3, 'delta_loss': -0.0011508241295814514, 'delta_accuracy': 0.0},
 {'unit': 9,
  'delta_loss': -0.0007780566811561584,
  'delta_accuracy': -0.005714297294616699}]

## 3. Candidate intervention and acceptance gate

We test the least damaging unit ablation on validation data. The gate is deliberately conservative: accept only if validation loss does not worsen beyond a small tolerance and accuracy does not drop beyond a specified tolerance.

For real work, use multiple seeds, task-specific safety tests, calibration checks, and confidence intervals.

In [5]:
candidate_unit = unit_effects_sorted[0]["unit"]
candidate = ablate_unit(baseline, candidate_unit)

candidate_val = evaluate(candidate, x_val, y_val)
candidate_test = evaluate(candidate, x_test, y_test)

MAX_VALIDATION_LOSS_INCREASE = 0.002
MAX_VALIDATION_ACCURACY_DROP = 0.002

accepted = (
    candidate_val["loss"] <= baseline_val["loss"] + MAX_VALIDATION_LOSS_INCREASE
    and candidate_val["accuracy"] >= baseline_val["accuracy"] - MAX_VALIDATION_ACCURACY_DROP
)

print("Candidate unit:", candidate_unit)
print("Baseline validation:", baseline_val)
print("Candidate validation:", candidate_val)
print("Candidate accepted:", accepted)
print("Candidate test (report only):", candidate_test)

Candidate unit: 28
Baseline validation: {'loss': 0.07294617593288422, 'accuracy': 0.9785714149475098}
Candidate validation: {'loss': 0.06656203418970108, 'accuracy': 0.9800000190734863}
Candidate accepted: True
Candidate test (report only): {'loss': 0.08601894229650497, 'accuracy': 0.972000002861023}


## 4. Compare weight perturbation with functional behavior

A small numerical perturbation is not automatically harmless. Evaluate it. Here we perturb one tensor in a disposable copy and measure the result.

In [6]:
def perturb_parameter(model, parameter_name, sigma=1e-3, seed=99):
    candidate = copy.deepcopy(model)
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    with torch.no_grad():
        for name, parameter in candidate.named_parameters():
            if name == parameter_name:
                noise = torch.randn(
                    parameter.shape,
                    generator=generator,
                    device=parameter.device,
                    dtype=parameter.dtype
                ) * sigma
                parameter.add_(noise)
                return candidate
    raise KeyError(parameter_name)

perturbed = perturb_parameter(baseline, "fc1.weight", sigma=1e-3)
print("Baseline:", evaluate(baseline, x_val, y_val))
print("Perturbed:", evaluate(perturbed, x_val, y_val))

Baseline: {'loss': 0.07294617593288422, 'accuracy': 0.9785714149475098}
Perturbed: {'loss': 0.07297898083925247, 'accuracy': 0.9785714149475098}


## 5. Save an experiment manifest

Every candidate should be reproducible and traceable. Store the source hash, candidate hash, intervention, validation measurements, and acceptance decision.

In [7]:
def model_hash(model):
    h = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        h.update(name.encode())
        h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

manifest = {
    "experiment": "hidden_unit_counterfactual_ablation",
    "seed": SEED,
    "source_model_sha256": model_hash(baseline),
    "candidate_model_sha256": model_hash(candidate),
    "intervention": {"type": "zero_outgoing_hidden_unit", "unit": candidate_unit},
    "baseline_validation": baseline_val,
    "candidate_validation": candidate_val,
    "candidate_test": candidate_test,
    "accepted": bool(accepted),
    "acceptance_policy": {
        "max_validation_loss_increase": MAX_VALIDATION_LOSS_INCREASE,
        "max_validation_accuracy_drop": MAX_VALIDATION_ACCURACY_DROP,
    },
    "warning": "Synthetic demonstration only; acceptance does not imply production readiness."
}

Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/experiment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

if accepted:
    torch.save(candidate.state_dict(), "artifacts/accepted_candidate_state_dict.pt")

print(json.dumps(manifest, indent=2))

{
  "experiment": "hidden_unit_counterfactual_ablation",
  "seed": 23,
  "source_model_sha256": "43406eddb6b04c23253561317614336b8a07845fec74c5344f66ee9baf8dc67f",
  "candidate_model_sha256": "6558a4c3aba33543b8e6893073f99744a742d8fc5dcf841f6ae05e8592066d10",
  "intervention": {
    "type": "zero_outgoing_hidden_unit",
    "unit": 28
  },
  "baseline_validation": {
    "loss": 0.07294617593288422,
    "accuracy": 0.9785714149475098
  },
  "candidate_validation": {
    "loss": 0.06656203418970108,
    "accuracy": 0.9800000190734863
  },
  "candidate_test": {
    "loss": 0.08601894229650497,
    "accuracy": 0.972000002861023
  },
  "accepted": true,
  "acceptance_policy": {
    "max_validation_loss_increase": 0.002,
    "max_validation_accuracy_drop": 0.002
  },
  "warning": "Synthetic demonstration only; acceptance does not imply production readiness."
}


## 6. Extension path toward LLMs

For a Transformer or LLM, preserve the same experiment contract but replace the MLP-specific intervention with architecture-aware adapters:

- Attention-head ablation or activation patching.
- MLP-neuron / channel interventions.
- Projection-matrix perturbations.
- Layer-wise or block-wise interventions.
- Weight-space alignment where mathematically valid.
- Token-level and task-specific behavioral suites.
- Long-context, multilingual, adversarial, and safety regression tests.

Do not assume a low-impact component on one benchmark is globally unnecessary. Measure across a declared evaluation suite and report uncertainty.

## Completion checklist

- [ ] Baseline is reproducible and hashed.
- [ ] Train/validation/test data are separated.
- [ ] Candidate edits occur on copies, never the source artifact.
- [ ] Evaluation includes both loss and task behavior.
- [ ] Acceptance thresholds are explicit.
- [ ] Manifest records the intervention and result.
- [ ] Human approval is required before deployment.

### Research connections
- [Git Re-Basin](https://arxiv.org/abs/2209.04836)
- [NFN implementation](https://github.com/AllanYangZhou/nfn)
- [Weight Oracles](https://arxiv.org/pdf/2610.07334)
- [TQCompressor](https://arxiv.org/abs/2401.16367)

## 7. Customer-facing demo: Model Assurance Gate

This notebook's experiment loop can become a demonstrable CI/CD gate for model artifacts.

### Example lifecycle

1. A model artifact is produced by training, fine-tuning, merging, or an external supplier.
2. The assurance job calculates identity and provenance evidence.
3. Architecture-aware inspection runs.
4. Behavioral and security probes execute.
5. Candidate changes are compared with the approved baseline.
6. Policy evaluates explicit thresholds.
7. The pipeline emits `PASS`, `REVIEW`, or `BLOCK` with evidence.

**Important:** `BLOCK` should require a defined policy violation or strong validated evidence. Novel anomaly scores should normally route to `REVIEW` until calibrated.

## 8. Demo scenario — secure model promotion

A useful initial demo can be run locally with the synthetic MLP in this notebook.

| Stage | Demonstration | Evidence |
|---|---|---|
| Intake | Load baseline and candidate | Hashes, metadata |
| Integrity | Compare state dictionaries | Changed tensor list |
| Behavior | Run fixed validation suite | Loss, accuracy, agreement |
| Counterfactual | Ablate selected unit | Delta metrics |
| Regression | Evaluate held-out test data | Final metrics |
| Gate | Apply explicit policy | PASS / REVIEW / BLOCK |
| Audit | Save manifest | Reproducible JSON record |

For the commercial demonstration, show the same workflow in a small FastAPI service or CI job after the Python package APIs stabilize.

## 9. Demo scenario — model supply-chain assessment

Use two checkpoints:
- a trusted baseline produced by your own training pipeline
- a candidate artifact obtained from a different build or source

Demonstrate:
- hash and metadata comparison
- unexpected tensor changes
- parameter distribution shifts
- architecture coverage
- behavioral probe differences
- evidence report and review decision

This is a credible AI security product demo even before learned Weight Oracle methods scale to billion-parameter models. The value is a reproducible evidence workflow, not a claim that weight statistics alone detect every malicious model.

## 10. Product metrics and acceptance criteria

Track product value with metrics that customers can understand.

| Product area | Measures |
|---|---|
| Security assessment | artifact coverage, known-issue detection, false-positive rate, time to triage |
| Model change assurance | regression detection, reproducibility, promotion / rollback time |
| Model optimization | quality retained, parameter count, memory, latency, inference cost |
| Model discovery | retrieval precision@k, task success after verification |
| Platform integration | API latency, pipeline runtime, supported architecture coverage |

Every benchmark report should include model version, dataset version, random seeds, hardware, software dependencies, and evaluation configuration.

## 11. Commercial positioning and boundaries

### Initial product proposition
**Model Assurance and Optimization Workbench:** inspect model artifacts, compare versions, investigate suspicious changes, measure functional impact, and validate optimization candidates before deployment.

### Initial target users
- ML platform and MLOps teams
- AI security / product security teams
- Organizations consuming third-party or open-weight models
- Teams operating model registries and production inference services

### What we can credibly demo first
- Reproducible artifact inventory and integrity evidence
- Model comparison with functional probes
- Sensitivity and counterfactual experiments on supported architectures
- CI-style promotion gate with machine-readable evidence
- Modular path to NFN, Weight Oracle, WeightCLIP, and generative weight methods

### What remains research
- Reliable raw-weight interpretation at frontier-model scale
- General-purpose backdoor detection from weights alone
- Automatic identification of globally irrelevant parameters
- Safe autonomous editing of arbitrary production LLM weights
- Universal architecture-independent weight alignment

The commercial advantage is a useful assurance workflow today, with a research roadmap that can add stronger weight-space intelligence as the methods mature.

## 12. Evaluate knowledge-derived graph artifacts counterfactually

This extension adds a controlled comparison for graph-derived computation while preserving the notebook's existing baseline/candidate discipline. We create a synthetic relational dataset in which a node's label depends partly on its own features and partly on a connected neighbor. A node-only model is the baseline; a graph message-passing model is the candidate.

The train/validation/test node partitions are disjoint. The graph structure is fixed and generated from a deterministic seed. This is a proof-of-mechanism—not evidence that a graph module improves a production LLM.

In [8]:
# Graph-derived model evaluation: node-only baseline vs message-passing candidate
import copy, hashlib, json
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(123)
N, FEAT = 180, 6
gen = torch.Generator().manual_seed(123)
Xg = torch.randn(N, FEAT, generator=gen)

# Ring + seeded local edges; symmetric adjacency with self-loops.
Ag = torch.zeros(N, N)
for i in range(N):
    j = (i + 1) % N
    Ag[i, j] = Ag[j, i] = 1.0
edge_rng = torch.rand(N, N, generator=gen)
extra = torch.triu((edge_rng < 0.025).float(), diagonal=1)
Ag = torch.maximum(Ag, extra + extra.T)
Ag = Ag + torch.eye(N)
deg = Ag.sum(dim=1).clamp_min(1.0)
Dg = torch.diag(deg.pow(-0.5))
A_norm_g = Dg @ Ag @ Dg

# Label depends on own signal and neighborhood signal.
neighbor_signal = A_norm_g @ Xg[:, 0]
score_g = 0.75 * Xg[:, 0] + 0.90 * neighbor_signal + 0.25 * Xg[:, 1]
yg = (score_g > score_g.median()).long()

perm = torch.randperm(N, generator=gen)
train_idx, val_idx, test_idx = perm[:108], perm[108:144], perm[144:]
assert len(set(train_idx.tolist()) & set(test_idx.tolist())) == 0

class NodeOnlyClassifier(nn.Module):
    def __init__(self, in_dim=FEAT, hidden=24):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 2))
    def forward(self, x, adjacency=None):
        return self.net(x)

class GraphMessageClassifier(nn.Module):
    def __init__(self, in_dim=FEAT, hidden=24):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden)
        self.head = nn.Linear(hidden, 2)
    def forward(self, x, adjacency):
        h = F.relu(self.proj(adjacency @ x))
        return self.head(h)

def fit_graph_experiment(model, use_graph, epochs=160):
    opt = torch.optim.Adam(model.parameters(), lr=0.025, weight_decay=1e-4)
    for _ in range(epochs):
        model.train()
        logits = model(Xg, A_norm_g) if use_graph else model(Xg)
        loss = F.cross_entropy(logits[train_idx], yg[train_idx])
        opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def graph_metrics(model, use_graph, indices):
    model.eval()
    logits = model(Xg, A_norm_g) if use_graph else model(Xg)
    pred = logits.argmax(dim=1)
    return {
        "accuracy": float((pred[indices] == yg[indices]).float().mean()),
        "loss": float(F.cross_entropy(logits[indices], yg[indices])),
    }

torch.manual_seed(202)
node_baseline = fit_graph_experiment(NodeOnlyClassifier(), use_graph=False)
torch.manual_seed(202)
graph_candidate = fit_graph_experiment(GraphMessageClassifier(), use_graph=True)

comparison = {
    "node_only_baseline": {
        "validation": graph_metrics(node_baseline, False, val_idx),
        "test": graph_metrics(node_baseline, False, test_idx),
    },
    "graph_message_passing_candidate": {
        "validation": graph_metrics(graph_candidate, True, val_idx),
        "test": graph_metrics(graph_candidate, True, test_idx),
    },
}
print(json.dumps(comparison, indent=2))
assert all(0.0 <= v["accuracy"] <= 1.0
           for model_result in comparison.values()
           for v in model_result.values())

{
  "node_only_baseline": {
    "validation": {
      "accuracy": 0.9722222089767456,
      "loss": 0.16856509447097778
    },
    "test": {
      "accuracy": 0.7777777910232544,
      "loss": 1.1060951948165894
    }
  },
  "graph_message_passing_candidate": {
    "validation": {
      "accuracy": 0.75,
      "loss": 0.8227918148040771
    },
    "test": {
      "accuracy": 0.5277777910232544,
      "loss": 2.0928568840026855
    }
  }
}


### Candidate evidence, retention gate, and artifact lineage

Do not select a candidate on test results. The example reports validation and test separately for demonstration; a production pipeline would select thresholds and hyperparameters on validation only, freeze the candidate, then open the held-out test set once.

The acceptance gate below is intentionally simple: require a validation improvement and no material test regression. Real gates should include confidence intervals, multiple seeds, source/entity/time-separated splits, calibration, robustness, security, latency, memory, and retained performance on the original model benchmark.

In [9]:
def state_dict_fingerprint(model):
    h = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        h.update(name.encode())
        h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

base_val = comparison["node_only_baseline"]["validation"]
cand_val = comparison["graph_message_passing_candidate"]["validation"]
base_test = comparison["node_only_baseline"]["test"]
cand_test = comparison["graph_message_passing_candidate"]["test"]

validation_gain = cand_val["accuracy"] - base_val["accuracy"]
test_delta = cand_test["accuracy"] - base_test["accuracy"]
acceptance = {
    "validation_accuracy_improved": validation_gain > 0.0,
    "test_accuracy_not_regressed_by_more_than_2pp": test_delta >= -0.02,
    "candidate_sha256": state_dict_fingerprint(graph_candidate),
    "baseline_sha256": state_dict_fingerprint(node_baseline),
    "validation_accuracy_delta": validation_gain,
    "test_accuracy_delta": test_delta,
    "decision": "eligible_for_further_review"
        if validation_gain > 0 and test_delta >= -0.02
        else "reject_or_revise",
}
print(json.dumps(acceptance, indent=2))
print("Important: this synthetic experiment is not a production promotion authorization.")

{
  "validation_accuracy_improved": false,
  "test_accuracy_not_regressed_by_more_than_2pp": false,
  "candidate_sha256": "a2d98350ca7ed2b2d7a3bb1a245e299989425e2f7390a159a692616952511325",
  "baseline_sha256": "11b88f738cad5d8aaf1ae2aa9abe0c73e1ff9aff1f52b491aaf392704a8e7d1d",
  "validation_accuracy_delta": -0.2222222089767456,
  "test_accuracy_delta": -0.25,
  "decision": "reject_or_revise"
}
Important: this synthetic experiment is not a production promotion authorization.


### Required evaluation matrix for future LLM-facing KDAE experiments

| Candidate path | What changes | Primary evidence |
|---|---|---|
| Base LLM | Nothing | Frozen benchmark baseline |
| Base + graph retrieval | Context only | Factual precision/recall, citation fidelity, latency |
| Base + graph encoder/projection | Compatible neural module | Held-out relation/entity generalization, retention |
| Base + graph-derived LoRA | Adapter parameters | Adapter delta, task success, retention, safety |
| Graph retrieval + LoRA | Context and parameters | Ablation against each component and combined path |

Use entity-disjoint and source-disjoint splits to prevent memorization leakage. Keep a separate provenance report: capability probes can estimate what a model appears to know, but weights alone generally cannot reveal the exact training corpus. Training-data claims require manifests, dataset lineage, checkpoint logs, and adapter history.